# 🏢 企业内部 API Agent 完整教程

本教程参照 **RAGFlow 代码组织方式**，从 0 开始教你构建一个调用企业内部 API 的 AI Agent 系统。

## 📁 项目结构

```
company_agent/                    ← 项目根目录
├── tools/                        ← 工具层（对应 RAGFlow: agent/tools/）
│   ├── base.py                   #   BaseTool 抽象基类 + ToolRegistry 注册表
│   ├── employees.py              #   GetEmployees 员工查询工具
│   ├── projects.py               #   GetProjects 项目查询工具
│   ├── metrics.py                #   GetMetrics 公司指标工具
│   └── search.py                 #   SearchCompany 内部搜索工具
├── llm/                          ← LLM 层（对应 RAGFlow: rag/llm/）
│   └── client.py                 #   LLMClient 封装 DeepSeek-V4-Flash
├── prompts/                      ← 提示词层（对应 RAGFlow: rag/prompts/）
│   ├── system.md                 #   系统提示词
│   └── next_step.md              #   下一步决策提示词
├── agent/                        ← Agent 层（对应 RAGFlow: agent/component/）
│   └── agent.py                  #   AgentWithTools 推理循环
└── main.py                       ← 运行入口
```

## 🎯 学习目标
1. 理解 RAGFlow 风格的分层架构设计
2. 掌握 Function Calling 工具注册与执行机制
3. 实现思维链（Chain of Thought）驱动的 Agent 推理
4. 实现多 API 调用的并行执行
5. 学会扩展新工具

## 🔗 对应 RAGFlow 源码路径

| 本教程步骤 | RAGFlow 对应代码 |
|-----------|-----------------|
| Step 1: 工具基类 | agent/tools/ (基础部分) |
| Step 2: 工具实现 | agent/tools/tavily.py:101-154 |
| Step 3: 思维链提示词 | rag/prompts/next_step.md (全文) |
| Step 4: LLM 客户端 | rag/llm/chat_model.py:1784-1814 |
| Step 5: 并行执行 | rag/llm/chat_model.py:519-552 |
| Step 6: Agent 推理 | agent/component/agent_with_tools.py:194-212 |
| Step 7: 完整运行 | 综合以上所有模块 |

## 🔑 使用的大模型
- **模型**: deepseek-v4-flash
- **API 端点**: `https://token-plan.cn-beijing.maas.aliyuncs.com/compatible-mode/v1`
- **兼容协议**: OpenAI Compatible

---
## 📦 Step 0: 环境准备

首先安装必要的依赖库。

In [ ]:
# 安装依赖
!pip install openai httpx nest_asyncio 2>&1 | tail -5

In [ ]:
import json
import asyncio
import os
import sys
import time
from typing import Any, Dict, List, Optional, Tuple

print("✅ 基础环境准备完成")


---
## 🏗️ Step 1: 实现工具基类与注册机制

> **对应 RAGFlow**: `agent/tools/` (基础部分)

### 架构说明

RAGFlow 使用工具注册表模式来管理所有可用工具。每个工具必须提供：
1. **name** — 工具的唯一标识符，LLM 通过此名称调用工具
2. **description** — 告诉 LLM 工具的功能
3. **parameters** — JSON Schema 格式，描述工具需要的参数
4. **__call__** — 工具的实际执行代码

这种设计的精妙之处在于：**工具定义与执行分离**。定义部分告诉 LLM "你能做什么"，执行部分完成实际工作。

In [ ]:
# 查看工具基类源码
from company_agent.tools.base import BaseTool, ToolRegistry
import inspect

print("=== BaseTool 抽象方法 ===")
for name, method in inspect.getmembers(BaseTool, predicate=inspect.isfunction):
    if not name.startswith("_"):
        sig = inspect.signature(method)
        print(f"  {name}{sig}")

print("\n=== ToolRegistry 方法 ===")
for name, method in inspect.getmembers(ToolRegistry, predicate=inspect.isfunction):
    if not name.startswith("_"):
        sig = inspect.signature(method)
        print(f"  {name}{sig}")

print("\n✅ 工具基类已实现")

In [ ]:
# 查看 to_openai_tool() 方法实现
import inspect
source = inspect.getsource(BaseTool.to_openai_tool)
print("BaseTool.to_openai_tool() 源码:")
print(source)

### 💡 关键设计点

- **to_openai_tool() 方法**: 将 Python 工具类转换为 OpenAI Function Calling 的 JSON 格式。这是 LLM 理解工具能力的桥梁。
- **注册表模式**: 用字典存储工具，支持按名称快速查找。RAGFlow 实际使用了更复杂的注册机制（支持动态加载）。
- **抽象基类**: 强制子类实现 name, description, parameters, __call__，保证所有工具接口一致。

---
## 🔧 Step 2: 实现企业内部 API 工具

> **对应 RAGFlow**: `agent/tools/tavily.py:101-154`

### 架构说明

每个工具是一个独立的 Python 文件，继承 `BaseTool`。本教程实现 4 个工具：

| 工具类 | 文件 | 对应 API |
|--------|------|----------|
| GetEmployees | `tools/employees.py` | `/api/employees` |
| GetProjects | `tools/projects.py` | `/api/projects` |
| GetMetrics | `tools/metrics.py` | `/api/metrics` |
| SearchCompany | `tools/search.py` | `/api/search` |

In [ ]:
import json
# 查看所有工具的源码结构
from company_agent.tools.employees import GetEmployees
from company_agent.tools.projects import GetProjects
from company_agent.tools.metrics import GetMetrics
from company_agent.tools.search import SearchCompany

tools_info = [
    ("GetEmployees", GetEmployees, "/api/employees"),
    ("GetProjects", GetProjects, "/api/projects"),
    ("GetMetrics", GetMetrics, "/api/metrics"),
    ("SearchCompany", SearchCompany, "/api/search"),
]

print("="*70)
print(f"{'工具类':<20} {'API 端点':<25} {'参数数量':>8}")
print("="*70)

for name, cls, api_path in tools_info:
    instance = cls()
    n_params = len(instance.parameters.get("properties", {}))
    print(f"{name:<20} {api_path:<25} {n_params:>8}")

print("="*70)

In [ ]:
import json
# 查看单个工具的完整定义
import inspect
source = inspect.getsource(GetEmployees)
print("=== GetEmployees 源码 ===")
print(source[:1200] + "...\n")

emp = GetEmployees()
print(f"name: {emp.name}")
print(f"description: {emp.description[:80]}...")
print(f"parameters: {json.dumps(emp.parameters, indent=2)}")

In [ ]:
# 创建注册表并注册所有工具
from company_agent.main import create_registry

API_BASE_URL = "http://localhost:8080"
registry = create_registry(api_base_url=API_BASE_URL)

print(f"✅ 已注册 {len(registry.get_all())} 个工具")
for tool in registry.get_all():
    print(f"   📝 {tool.name}: {tool.description[:50]}...")

In [ ]:
# 查看工具转换为 OpenAI 格式后的样子
tools = registry.to_openai_tools()
print(f"转换后共 {len(tools)} 个工具定义")
print("\n第一个工具定义示例:")
print(json.dumps(tools[0], indent=2))

In [ ]:
# 测试：直接调用每个工具验证 API 连通性
async def test_all_tools():
    print("🧪 测试所有工具调用...\n")
    
    for name in ["get_employees", "get_projects", "get_metrics"]:
        tool = registry.get(name)
        result = await tool()
        print(f"--- {name} ---")
        print(result[:250])
        print()
    
    search = registry.get("search_company")
    result = await search(query="ai")
    print(f"--- search_company(query='ai') ---")
    print(result)

await test_all_tools()

In [ ]:
# 测试带参数的调用
async def test_tools_with_args():
    print("🧪 测试带参数的工具调用...\n")
    
    emp = registry.get("get_employees")
    result = await emp(department="工程部")
    print(f"--- get_employees(department='工程部') ---")
    print(result)
    
    proj = registry.get("get_projects")
    result = await proj(status="进行中")
    print(f"\n--- get_projects(status='进行中') ---")
    print(result)

await test_tools_with_args()

### 💡 关键设计点

- **JSON Schema 参数定义**: parameters 属性定义了工具参数的 JSON Schema。LLM 的 function calling 机制会根据这个 schema 自动验证参数。
- **异步 __call__**: 使用 async def 让工具支持异步执行，这是实现并行搜索的关键。
- **结果格式化**: 每个工具将 API 返回的结构化数据转换为 LLM 友好的文本格式。
- **api_base_url 可配置**: 构造函数接受 api_base_url 参数，支持不同环境部署。

---
## 🧠 Step 3: 实现思维链提示词

> **对应 RAGFlow**: `rag/prompts/next_step.md` (全文)

### 架构说明

思维链（Chain of Thought）提示词是 Agent 智能决策的核心。它告诉 LLM：
1. 当前处于什么状态
2. 已经获得了什么信息
3. 下一步应该做什么

RAGFlow 的 next_step.md 提示词指导 LLM 在每个推理步骤中进行思考。

**设计精妙之处**: 提示词使用独立的 `.md` 文件管理，而非硬编码在 Python 代码中。这样的好处是：
- 方便非技术人员修改提示词
- 版本控制清晰
- 支持多语言提示词

In [ ]:
# 查看提示词文件
from company_agent.prompts import SYSTEM_PROMPT, NEXT_STEP_PROMPT

print("=== 系统提示词 (prompts/system.md) ===")
print(SYSTEM_PROMPT)
print(f"\n...总长度: {len(SYSTEM_PROMPT)} 字符")

In [ ]:
print("=== 下一步决策提示词 (prompts/next_step.md) ===")
print(NEXT_STEP_PROMPT)

In [ ]:
# 测试提示词构建函数
from company_agent.prompts import build_next_step_prompt

# 第一次调用（无搜索历史）
prompt1 = build_next_step_prompt("工程部有多少人？")
print("=== 第一次调用（无搜索历史）===")
print(prompt1)

print("\n" + "="*60 + "\n")

# 第二次调用（带搜索历史）
prompt2 = build_next_step_prompt(
    "工程部有多少人？",
    search_history="已查询：get_employees(department='工程部') 返回 2 名员工"
)
print("=== 第二次调用（带搜索历史）===")
print(prompt2)

### 💡 关键设计点

- **NEXT_STEP_PROMPT 模板**: 使用 {user_query} 和 {search_history} 作为占位符，每次推理循环时动态填充。
- **build_next_step_prompt()**: 将状态信息编码为提示词，引导 LLM 思考。
- **搜索历史传递**: 将之前的搜索结果传递给 LLM，让它基于已有信息做决策 — 这就是思维链的核心机制。

---
## ⚡ Step 4: 实现 LLM 客户端

> **对应 RAGFlow**: `rag/llm/chat_model.py:1784-1814`（LiteLLM 并行）

### 架构说明

LLM 客户端封装了与大模型的对话逻辑。本教程使用 **DeepSeek-V4-Flash**，通过 OpenAI 兼容协议接入。

**API Key 传入的 3 种方式**：
1. 构造函数参数：`LLMClient(api_key="sk-xxx")`
2. 环境变量：`export DEEPSEEK_API_KEY="sk-xxx"`
3. 都不传 → 自动进入模拟模式

优先级：构造函数参数 > 环境变量 > 空字符串（模拟模式）

In [ ]:
# 查看 LLMClient 源码
import inspect
from company_agent.llm import LLMClient

print("=== LLMClient 方法 ===")
for name, method in inspect.getmembers(LLMClient, predicate=inspect.isfunction):
    if not name.startswith("_"):
        sig = inspect.signature(method)
        print(f"  {name}{sig}")

print("\n=== chat() 方法核心逻辑 ===")
source = inspect.getsource(LLMClient.chat)
# 显示关键部分
lines = source.split('\n')
for line in lines[:30]:
    print(line)

In [ ]:
from company_agent.llm import LLMClient
# 初始化 LLM 客户端
import os

# 方式 1: 使用环境变量（如果已设置）
# 方式 2: 直接传参（取消注释下方代码）
# llm = LLMClient(api_key="sk-your-key-here")

llm = LLMClient()
print(f"模型: {llm.model}")
print(f"端点: {llm.base_url}")
print(f"模式: {llm.mode}")

In [ ]:
from company_agent.llm import LLMClient
# 测试：调用 LLM 获取工具调用决策
async def test_llm_decision():
    messages = [
        {"role": "system", "content": "你是公司助手。"},
        {"role": "user", "content": "工程部有哪些员工？"}
    ]
    tools = registry.to_openai_tools()
    print(f"发送给 LLM 的工具数量: {len(tools)}")
    print(f"工具名称: {[t['function']['name'] for t in tools]}")
    
    text, tool_calls = await llm.chat(messages, tools=tools)
    
    if tool_calls:
        print(f"\n✅ LLM 决定调用 {len(tool_calls)} 个工具:")
        for tc in tool_calls:
            print(f"   - {tc['name']}({tc['arguments']})")
    else:
        print(f"\n✅ LLM 直接回答: {text[:100]}...")

await test_llm_decision()

---
## 🔀 Step 5: 实现并行执行器

> **对应 RAGFlow**: `rag/llm/chat_model.py:519-552`（基础并行）

### 架构说明

这是整个系统中最精妙的部分。RAGFlow 实现了并行执行：

```
LLM 输出 → [tool_call_1, tool_call_2, tool_call_3]
                ↓            ↓            ↓
         task_1运行    task_2运行    task_3运行  ← 同时执行！
                ↓            ↓            ↓
         result_1    result_2    result_3
```

实现并行执行的核心 Python 原语是 `asyncio.gather()`。

In [ ]:
import time
import asyncio
# 测试：手动并行执行多个工具
async def test_parallel_execution():
    tool_calls = [
        {"id": "call_1", "name": "get_employees", "arguments": {"department": "工程部"}},
        {"id": "call_2", "name": "get_projects", "arguments": {"status": "进行中"}},
        {"id": "call_3", "name": "get_metrics", "arguments": {}},
    ]
    
    print(f"⚡ 并行执行 {len(tool_calls)} 个工具调用...\n")
    start_time = time.time()
    
    async def execute_one(tc):
        tool = registry.get(tc["name"])
        result = await tool(**tc.get("arguments", {}))
        return {"tool_call_id": tc["id"], "name": tc["name"], "result": result}
    
    tasks = [execute_one(tc) for tc in tool_calls]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    elapsed = time.time() - start_time
    print(f"\n✅ 全部完成，耗时 {elapsed:.3f}s\n")
    
    for r in results:
        print(f"--- {r['name']} ---")
        print(r['result'][:150])
        print()

await test_parallel_execution()

### 💡 关键设计点

- **asyncio.gather(*tasks)**: 这是 Python 并发执行多个异步任务的标准方式。所有任务同时开始，等待最慢的那个完成。
- **并行 vs 串行**: 如果串行执行 3 个各需 0.5 秒的 API 调用，需要 1.5 秒；并行执行只需约 0.5 秒。
- **return_exceptions=True**: 防止一个任务失败导致所有任务取消。
- **RAGFlow 的实现**: 在 chat_model.py:519-552 中使用了类似的 asyncio.gather 模式。

---
## 🔄 Step 6: 实现 Agent 推理循环

> **对应 RAGFlow**: `agent/component/agent_with_tools.py:194-212`

### 架构说明

Agent 推理循环是整个系统的大脑。它实现了 **ReAct 模式**（Reasoning + Acting）：
```
思考 → 行动 → 观察 → 思考 → 行动 → 观察 → ... → 最终答案
```

RAGFlow 的 agent_with_tools.py:194-212 正是实现了这个循环。

In [ ]:
# 查看 Agent 类结构
import inspect
from company_agent.agent import AgentWithTools

print("=== AgentWithTools 方法 ===")
for name, method in inspect.getmembers(AgentWithTools, predicate=inspect.isfunction):
    if not name.startswith("_"):
        sig = inspect.signature(method)
        print(f"  {name}{sig}")

In [ ]:
# 查看 run() 方法的核心逻辑
source = inspect.getsource(AgentWithTools.run)
lines = source.split('\n')
print("=== AgentWithTools.run() 核心逻辑 ===")
for line in lines[:60]:
    print(line)

### 💡 关键设计点

- **循环结构**: `for step in range(1, max_steps + 1)` 实现了 ReAct 循环。
- **思维链传递**: 每次循环调用 `build_next_step_prompt()`，将搜索历史传入，让 LLM "记住"之前的搜索结果。
- **工具结果回传**: 将工具执行结果以 `role: "tool"` 格式添加回 messages，LLM 能看到这些结果。
- **终止条件**: 当 LLM 不再返回 tool_calls 时，说明它认为信息足够，返回最终答案。
- **安全机制**: `max_steps` 防止 LLM 陷入无限搜索循环。

---
## 🎬 Step 7: 完整运行示例

现在让我们把所有组件串联起来，运行完整的示例！

### 架构总览

```
┌─────────────────────────────────────────────────────────────┐
│                    AgentWithTools                           │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐  │
│  │  思维链提示词  │───▶│  LLM Client  │───▶│ 并行执行器    │  │
│  │ next_step.md │    │ chat()       │    │ asyncio.gather│  │
│  └──────────────┘    └──────┬───────┘    └──────┬───────┘  │
│                             │                   │          │
│                      tool_calls           执行结果           │
│                             │                   │          │
│                             ▼                   ▼          │
│                    ┌────────────────────────────────┐      │
│                    │        ToolRegistry            │      │
│                    │  ┌──────────┐ ┌──────────┐    │      │
│                    │  │ GetEmps  │ │ GetProj  │    │      │
│                    │  └──────────┘ └──────────┘    │      │
│                    │  ┌──────────┐ ┌──────────┐    │      │
│                    │  │ GetMetrs │ │ SearchCo │    │      │
│                    │  └──────────┘ └──────────┘    │      │
│                    └────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# 使用便捷函数运行 Agent
from company_agent.main import run
import nest_asyncio
try:
    nest_asyncio.apply()
except:
    pass

print("="*70)
print("示例 1: 项目 + 员工查询")
print("="*70)
query1 = "我们公司有哪些进行中的项目？工程部有哪些员工？"
result1 = await run(query1, api_base_url=API_BASE_URL, max_steps=3)
print(f"\n✅ 示例 1 完成！")

In [ ]:
print("="*70)
print("示例 2: 指标 + 项目状态查询")
print("="*70)
query2 = "公司 Q3 的营收情况如何？现在有哪些规划中的项目？"
result2 = await run(query2, api_base_url=API_BASE_URL, max_steps=3)
print(f"\n✅ 示例 2 完成！")

In [ ]:
print("="*70)
print("示例 3: 使用搜索工具")
print("="*70)
query3 = "公司关于 AI 的项目有哪些？"
result3 = await run(query3, api_base_url=API_BASE_URL, max_steps=3)
print(f"\n✅ 示例 3 完成！")

In [ ]:
# 示例 4: 使用 API Key 直接传入
print("="*70)
print("示例 4: 带 API Key 的运行")
print("="*70)

# 如果你有 API Key，取消注释下方代码：
# api_key = "sk-your-key-here"
# query4 = "市场部的员工有哪些？"
# result4 = await run(query4, api_key=api_key, api_base_url=API_BASE_URL, max_steps=3)
# print(f"\n✅ 示例 4 完成！")

print("(取消注释上方代码并填入 API Key 即可运行)")

---
## 📊 总结：代码组织与 RAGFlow 对照

### 文件映射关系

| 本教程模块 | 对应 RAGFlow 文件 | 行号 | 核心功能 |
|-----------|------------------|------|---------|
| BaseTool, ToolRegistry | agent/tools/base.py | 全文 | 工具基类和注册 |
| GetEmployees | agent/tools/employees.py | 全文 | 员工查询工具 |
| GetProjects | agent/tools/projects.py | 全文 | 项目查询工具 |
| GetMetrics | agent/tools/metrics.py | 全文 | 公司指标工具 |
| SearchCompany | agent/tools/search.py | 全文 | 内部搜索工具 |
| SYSTEM_PROMPT, NEXT_STEP_PROMPT | rag/prompts/*.md | 全文 | 思维链提示词 |
| LLMClient.chat() | rag/llm/chat_model.py | 1784-1814 | LiteLLM 并行 |
| execute_tools_parallel() | rag/llm/chat_model.py | 519-552 | 基础并行执行 |
| AgentWithTools.run() | agent/component/agent_with_tools.py | 194-212 | Agent 推理集成 |

### 三个核心问题的答案

1. **大模型如何实现工具调用？**
   - 工具定义（name, description, parameters）+ to_openai_tool() 转换
   - LLM 通过 function calling 机制决定调用哪个工具
   - ToolRegistry 查找并执行对应工具

2. **思维链 + 多任务并行执行的代码？**
   - 思维链：next_step.md 提示词 + 搜索历史传递
   - 并行执行：使用 asyncio.gather() 并发执行多个工具
   - 关键：LLM 一次返回多个 tool_calls，执行器并发运行

3. **LLM 思维链如何做工具调用决策？**
   - 系统提示词定义工具使用说明
   - 每步提示词包含当前状态（已有信息 + 用户问题）
   - LLM 基于这些信息自主决定：是否需要调用 → 调用什么 → 是否继续
   - 多轮迭代中，决策基于逐步积累的信息（这就是思维链的力量）

### 关键设计模式

1. **注册表模式**: 工具通过注册表管理，支持动态扩展
2. **工厂模式**: to_openai_tool() 将工具转换为 LLM 可理解的格式
3. **策略模式**: 思维链提示词定义了推理策略，LLM 执行具体推理
4. **异步并行**: 使用 asyncio.gather 实现真正的并行 API 调用
5. **ReAct 循环**: 思考→行动→观察的迭代推理模式

### 如何扩展

**添加新 API 工具（3 步）**：

1. 在 `tools/` 目录下创建新文件，继承 `BaseTool`，实现 `name`, `description`, `parameters`, `__call__` 四个方法
2. 在 `main.py` 的 `create_registry()` 中注册
3. 完成！LLM 会自动发现新工具并决定是否需要调用它

**切换 LLM 模型**：

```python
llm = LLMClient(
    api_key="your-key",
    model="gpt-4o",
    base_url="https://api.openai.com/v1"
)
```

**修改提示词**：

直接编辑 `prompts/system.md` 或 `prompts/next_step.md`，无需修改 Python 代码。